In [5]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb
import time
from itertools import combinations
from IPython.display import display

        Introducción

El presente trabajo se basa en el conjunto de datos del censo de vivienda de California de 1990, el cual recopila información demográfica, estructural y geoespacial a nivel de bloque censal. El conjunto de datos permite analizar las características urbanas y socioeconómicas de distintas zonas de California durante la década de los 90, integrando variables como la ubicación geográfica (longitud y latitud), la antigüedad de las construcciones, la población y los ingresos medios.

Objetivo Principal: Predecir de manera precisa el valor mediano de las viviendas (median_house_value) en un determinado sector o bloque censal.

Tipo de Problema: Aprendizaje supervisado de regresión con variable continua (en dólares estadounidenses).

        Descripción del conjunto de datos

Número de observaciones: 20,640 registros.

Número de variables: 10 variables en total, divididas en 9 variables predictoras y 1 variable objetivo.

Significado de la variable objetivo (median_house_value): Representa el valor mediano de la vivienda en un bloque censal específico, expresado en dólares estadounidenses (USD). (Presenta un límite superior o censura en exactamente $500,001, correspondientes a las viviendas que superaban dicho monto).

Tipos de variables:
9 variables de tipo numérico flotante (float64) correspondientes a coordenadas, mediciones estructurales y variables socioeconómicas.
1 variable de tipo categórico de texto (string) que describe la proximidad de la vivienda al mar (ocean_proximity).

-longitude (Numérica, float): Coordenada geográfica que indica la longitud; a menor valor, más al occidente se ubica el sector.

-latitude (Numérica, float): Coordenada geográfica que indica la latitud; a mayor valor, más al norte se ubica el sector.

-housing_median_age (Numérica, float): Antigüedad mediana de las construcciones en el bloque censal (en años).

-total_rooms (Numérica, float): Cantidad total de habitaciones registradas en el bloque. (Aunque corresponde a valores realmente enteros, se almacenan como decimales para facilitar y dar mayor precisión a los cálculos).

-total_bedrooms (Numérica, float): Cantidad total de dormitorios en el bloque censal (207 valores faltantes (equivalente al 1% de los registros de la columna).).

-population (Numérica, float): Cantidad total de residentes que habitan en el bloque.

-households (Numérica, float): Cantidad total de hogares o unidades familiares en el sector.

-median_income (Numérica, float): Ingreso mediano de los hogares en el bloque censal en decenas de miles de dólares 

-median_house_value (Numérica, float — Variable Objetivo): Valor mediano de la vivienda en el bloque (en USD).

-ocean_proximity (Categórica, string): Ubicación de la vivienda con respecto al mar u océano (ej. <1H OCEAN, INLAND, NEAR OCEAN, NEAR BAY, ISLAND).

        Posibles limitaciones del conjunto de datos:

-Al tratarse de datos del censo estadounidense de 1990, no reflejan el mercado actual; pero son idóneas para el contexto académico del proyecto

-La variable total_bedrooms cuenta con 207 registros nulos, por lo que se necesita un análisis y una estrategia de imputación antes de entrenar cualquier modelo.

-Múltiples variables predictoras y la propia variable objetivo muestran distribuciones sesgadas y valores extremos o de amplio rango, lo cual se debe tener en cuenta al momento de definir las transformaciones o seleccionar algoritmos tolerantes a estas características.

        Estadísticas descriptivas:

Las variables numéricas reflejan rangos muy amplios; por ejemplo, la población por bloque va desde 3 hasta 35,682 habitantes, y el ingreso mediano (median_income) oscila entre 0.4999 y 15.0001 (en decenas de miles de dólares). Para la variable objetivo (median_house_value), se registra una media de $206,856 USD y una mediana de $179,700 USD, con un rango que parte desde los $14,999 USD hasta el tope de $500,001 USD.

        Distribución de la variable objetivo (median_house_value):

La distribución presenta una marcada asimetría a la derecha. La mayor parte de los valores de las viviendas se concentran entre los $100,000 y $250,000 USD.
Se identifica un límite superior muy marcado en $500,001 USD (con alrededor de 1,000 registros). Esto se debe al censo de 1990.

        Análisis de correlación entre variables numéricas:

median_income presenta la correlación lineal más alta y positiva con el valor de la vivienda (0.69), indicando que el ingreso del bloque censal es el predictor lineal individual más determinante. El resto de las variables numéricas muestran correlaciones lineales relativamente bajas o moderadas con el precio. Por lo que las relaciones no son lineales, lo que nos inclina al uso de modelos basados en árboles. Por otra parte, hay una alta  multicolinealidad entre las variables total_rooms, total_bedrooms, population y households, con coeficientes de correlación de Pearson muy altos entre sí.

        Análisis de las relaciones entre variables predictoras y la variable objetivo:

-Relación geográfica (Longitud y Latitud): Las representaciones geográficas de California demuestran que el valor mediano de la vivienda es considerablemente más alto en zonas costeras. 

-median_income vs. median_house_value: Muestra una relación positiva evidente, acompañada de heterocedasticidad a mayores niveles de ingreso y una concentración persistente de propiedades en el tope de $500,000 USD a lo largo de diversos tramos de ingresos.

-ocean_proximity vs. median_house_value: Los diagramas de caja (boxplots) confirman que las viviendas más costosas se concentran cerca del mar. Se observa que las zonas clasificadas como INLAND concentran las medianas de valor más bajas y un rango acotado, mientras que categorías como <1H OCEAN, NEAR BAY y NEAR OCEAN presentan distribuciones con los bigotes extendidos hacia los límites superior e inferior, además de una alta densidad de valores extremos en los límites superiores de precio.